# SRQ-FLY Priority 2D — CIFAR-100 train-only equivalence
This is the last pre-test gate. It compares Priority 2B and the selected Priority 2C backend on real frozen training features. `test.pt` must remain absent.

In [ ]:
# Edit path/source values only.
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH='experiment/soho-selfcontained'
WORK_DIR='/content/SOHO-CL'
FEATURE_CACHE_DIR='/content/srq_priority2d_cifar_features'
WTA_CACHE_DIR='/content/srq_priority2d_wta_10000'
OUTPUT_DIR='/content/srq_priority2d_output'
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Fresh clone, dependencies, GPU and immutable source check.
import hashlib,json,os,shutil,subprocess,sys,time
from pathlib import Path
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub','pandas'],check=True)
import torch
assert torch.cuda.is_available(),'Select Runtime -> Change runtime type -> T4 GPU'
CONFIG='configs/srq_fly_priority2d_cifar100_equivalence.json'
RUNNER='tools/srq_fly_priority2d_equivalence.py'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert sha(CONFIG)=='4e37da05c1d1c2adf2ba4ab74fd4a25aae1f616937f848bea6bd58d3b7411dd3'
assert sha(RUNNER)=='1dcfc5fd940c47fdb451d1647d04f16e981eaa114ed889f8bbc835ee1541970b'
assert sha('methods/srq_fly_optimized/learner.py')=='40edac2e2cc88faac549f5c87217f3143d815bf53ecad8a37dfdb22c112691ae'
assert sha('tools/srq_fly_priority1_ablation.py')=='b65ce01bfc2e2f9f07a61ecd056637156b504626c74c45b6aa3a7c8162e22136'
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Repository must be clean'
print('SOURCE CHECK PASS | commit',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),'| GPU',torch.cuda.get_device_name(0))

In [ ]:
# Correctness and protocol gate; synthetic/CPU only.
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_srq_fly_optimized.py','tests/test_srq_fly_priority2c_memory.py','tests/test_srq_fly_priority2d_equivalence.py'],check=True)
print('PRIORITY-2D CORRECTNESS GATE: PASS')

In [ ]:
# Download locked checkpoint and processed CIFAR-100 source.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==346284714
assert sha(CHECKPOINT_PATH)=='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
print('checkpoint:',CHECKPOINT_PATH)
print('CIFAR-100:',CIFAR_ROOT)

In [ ]:
# Extract TRAIN features to temporary Colab disk only; never materialize test.pt.
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256','32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b','--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_priority2d','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('TRAIN FEATURE EXTRACTION START — approximately one progress line per batch/task.',flush=True)
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and (cache/'metadata.json').is_file()
assert not (cache/'test.pt').exists(),'FAIL: held-out test cache became visible'
metadata=json.loads((cache/'metadata.json').read_text())
assert metadata['feature_dim']==768 and metadata['finite'] is True
print('TRAIN CACHE PASS:',metadata.get('train_shape'),'| test.pt absent')

In [ ]:
# Two isolated methods on the identical train/validation stream and WTA cache.
Path(OUTPUT_DIR).mkdir(parents=True,exist_ok=True)
command=[sys.executable,'-u',RUNNER,'run','--config',CONFIG,'--feature-cache-dir',FEATURE_CACHE_DIR,'--code-cache-dir',WTA_CACHE_DIR,'--output-dir',OUTPUT_DIR,'--device','cuda','--require-clean-git']
print('REAL TRAIN-ONLY EQUIVALENCE START: shared WTA cache, then 2 x 10 tasks.',flush=True)
completed=subprocess.run(command)
assert completed.returncode==0,'Priority-2D failed; return the complete traceback without editing gates.'
RESULT=Path(OUTPUT_DIR)/'priority2d_equivalence_results.json'
payload=json.loads(RESULT.read_text())
print('PRIORITY-2D DECISION:',payload['status'])
print(json.dumps(payload['gates'],indent=2))

In [ ]:
# Compact audit table.
import pandas as pd
rows=[]
for row in payload['results']:
    rows.append({'method':row['method'],'validation_AIA':row['validation_average_accuracy'],'update_seconds':row['total_update_seconds'],'peak_allocated_GiB':row['peak_cuda_allocated_bytes']/2**30,'peak_reserved_GiB':row['peak_cuda_reserved_bytes']/2**30,'state_MiB':row['persistent_state_bytes']/2**20,'solver_residual_max':row['maximum_solver_relative_residual']})
display(pd.DataFrame(rows))
print(json.dumps({'stage_accuracy_gap_pp':payload['maximum_stage_accuracy_gap_pp'],'probe_logit_drift':payload['relative_probe_logit_drift'],'update_ratio':payload['update_ratio_to_priority2b'],'peak_ratio':payload['peak_allocated_ratio_to_priority2b']},indent=2))
print('STOP and return the ZIP; no held-out test has been authorized here.')

In [ ]:
# Export evidence; large feature/WTA caches are deliberately excluded.
bundle=Path('/content/srq_fly_priority2d_train_only')
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
shutil.copy2(CONFIG,bundle/Path(CONFIG).name)
shutil.copytree(OUTPUT_DIR,bundle/'results')
(bundle/'repo_commit.txt').write_text(subprocess.check_output(['git','rev-parse','HEAD'],text=True))
archive=shutil.make_archive('/content/srq_fly_priority2d_train_only','zip',bundle.parent,bundle.name)
print('artifact:',archive,'sha256:',sha(archive))
from google.colab import files
files.download(archive)